In [28]:
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# split para modelado
from sklearn.model_selection import train_test_split
# Scaled | Escalado
from sklearn.preprocessing import StandardScaler, MinMaxScaler
# Encoding | Codificación
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
from sklearn.metrics import accuracy_score
# To save models
import math
import json
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestClassifier
# Feature Selection
from sklearn.feature_selection import f_classif, SelectKBest
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix
from pickle import dump
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor


In [29]:
df_prueba = "anthonny"
if df_prueba == "anthonny":
    df = pd.read_csv("../data/processed/df")
    df = df.sort_values("num_semana").reset_index(drop=True)

    weeks = df["num_semana"].unique()
    cut_w = int(len(weeks) * 0.8)

    train_weeks = weeks[:cut_w]
    test_weeks  = weeks[cut_w:]

    train = df[df["num_semana"].isin(train_weeks)]
    test  = df[df["num_semana"].isin(test_weeks)]

    X_train, y_train = train.drop(columns=["y"]), train["y"]
    X_test,  y_test  = test.drop(columns=["y"]),  test["y"]

else:
    df = pd.read_csv("../data/processed/df_ineta.cvs")
    df = df.sort_values("weekend").reset_index(drop=True) 

    X = df.drop(columns=["weekend"])
    y = df["weekend"]

    cut = int(len(df) * 0.8)
    X_train, X_test = X.iloc[:cut], X.iloc[cut:]
    y_train, y_test = y.iloc[:cut], y.iloc[cut:]
    print ("Trabajaremos con el DF de ineta")

In [30]:
cat_cols = X_train.select_dtypes(include=["object","category"]).columns
num_cols = X_train.columns.difference(cat_cols)

preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", "passthrough", num_cols)
])


In [31]:
model = Pipeline([
    ("prep", preprocess),
    ("rf", RandomForestRegressor(random_state=18))
])

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prep', ...), ('rf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains spar

In [32]:
pred_test = model.predict(X_test)
pred_train = model.predict(X_train)
mse_test  = mean_squared_error(y_test, pred_test)
rmse_test = np.sqrt(mse_test)
r2_test   = r2_score(y_test, pred_test)

print(f"MSE (Error cuadrático medio): {mse_test:.2f}")
print(f"RMSE (Raíz del ECM): {rmse_test:.2f} Cantidad media de error en cuanto a prediccion por producto")
print(f"R² (Coef. determinación): {r2_test:.2f} % de precision")


MSE (Error cuadrático medio): 24.51
RMSE (Raíz del ECM): 4.95 Cantidad media de error en cuanto a prediccion por producto
R² (Coef. determinación): 0.69 % de precision


In [33]:
model = Pipeline([
    ("prep", preprocess),
    ("rf", XGBRegressor(random_state=18))
])

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prep', ...), ('rf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains spar

In [34]:
pred_test = model.predict(X_test)
pred_train = model.predict(X_train)
mse_test  = mean_squared_error(y_test, pred_test)
rmse_test = np.sqrt(mse_test)
r2_test   = r2_score(y_test, pred_test)

print(f"MSE (Error cuadrático medio): {mse_test:.2f}")
print(f"RMSE (Raíz del ECM): {rmse_test:.2f} Cantidad media de error en cuanto a prediccion por producto")
print(f"R² (Coef. determinación): {r2_test:.2f} % de precision")


MSE (Error cuadrático medio): 33.17
RMSE (Raíz del ECM): 5.76 Cantidad media de error en cuanto a prediccion por producto
R² (Coef. determinación): 0.58 % de precision


In [35]:
model = Pipeline([
    ("prep", preprocess),
    ("rf", LGBMRegressor(random_state=18, verbosity=-1))
])

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prep', ...), ('rf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains spar

In [36]:
pred_test = model.predict(X_test)
pred_train = model.predict(X_train)
mse_test  = mean_squared_error(y_test, pred_test)
rmse_test = np.sqrt(mse_test)
r2_test   = r2_score(y_test, pred_test)

print(f"MSE (Error cuadrático medio): {mse_test:.2f}")
print(f"RMSE (Raíz del ECM): {rmse_test:.2f} Cantidad media de error en cuanto a prediccion por producto")
print(f"R² (Coef. determinación): {r2_test:.2f} % de precision")


MSE (Error cuadrático medio): 29.69
RMSE (Raíz del ECM): 5.45 Cantidad media de error en cuanto a prediccion por producto
R² (Coef. determinación): 0.62 % de precision


/home/codespace/.local/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [26]:
model = Pipeline([
    ("prep", preprocess),
    ("rf", CatBoostRegressor(random_state=18))
])

model.fit(X_train, y_train)

Learning rate set to 0.062265
0:	learn: 11.3945137	total: 49.7ms	remaining: 49.7s
1:	learn: 10.9673793	total: 53.1ms	remaining: 26.5s
2:	learn: 10.5705100	total: 55.9ms	remaining: 18.6s
3:	learn: 10.2016028	total: 59.3ms	remaining: 14.8s
4:	learn: 9.8756288	total: 62.8ms	remaining: 12.5s
5:	learn: 9.5542960	total: 65.6ms	remaining: 10.9s
6:	learn: 9.2679678	total: 68.4ms	remaining: 9.71s
7:	learn: 9.0019357	total: 71.2ms	remaining: 8.83s
8:	learn: 8.7490056	total: 74.1ms	remaining: 8.16s
9:	learn: 8.5120630	total: 77ms	remaining: 7.62s
10:	learn: 8.2975143	total: 79.9ms	remaining: 7.18s
11:	learn: 8.0965044	total: 82.7ms	remaining: 6.81s
12:	learn: 7.9167935	total: 85.9ms	remaining: 6.52s
13:	learn: 7.7518443	total: 88.7ms	remaining: 6.25s
14:	learn: 7.6030027	total: 91.5ms	remaining: 6.01s
15:	learn: 7.4695454	total: 94.3ms	remaining: 5.8s
16:	learn: 7.3411397	total: 97ms	remaining: 5.61s
17:	learn: 7.2207266	total: 99.8ms	remaining: 5.45s
18:	learn: 7.1115915	total: 103ms	remaining: 

AttributeError: The following error was raised: 'CatBoostRegressor' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order.

AttributeError: The following error was raised: 'CatBoostRegressor' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order.

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['product'], dtype='object')),
                                                 ('num', 'passthrough',
                                                  Index(['num_semana', 'y_lag1', 'y_lag2', 'y_lag3', 'y_lag4', 'y_lag5',
       'y_lag6', 'y_lag7', 'y_lag8'],
      dtype='object'))])),
                ('rf',
                 <catboost.core.CatBoostRegressor object at 0x7231fe688290>)])

In [27]:
pred_test = model.predict(X_test)
pred_train = model.predict(X_train)
mse_test  = mean_squared_error(y_test, pred_test)
rmse_test = np.sqrt(mse_test)
r2_test   = r2_score(y_test, pred_test)

print(f"MSE (Error cuadrático medio): {mse_test:.2f}")
print(f"RMSE (Raíz del ECM): {rmse_test:.2f} Cantidad media de error en cuanto a prediccion por producto")
print(f"R² (Coef. determinación): {r2_test:.2f} % de precision")

AttributeError: The following error was raised: 'CatBoostRegressor' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order.